# Noise Busters

> This is the open hackathon challenge of the Fall Fest 2026 error-mitigation track. The guided, graded exercise is the **SLC partner lab** (`SLC_magnetization_partner_lab.ipynb`). If these tools are new to you, work through that one first.

Are your expectation values coming out strange? A bias that stubbornly refuses to die, gate errors clouding your observables, the ghosts of readout error haunting every shot — you don't have to fear any of it anymore.

This is the time for you to be a Noise Buster now. Your proton pack is a Noise Learner, an Executor, and the SLC/PNA add-ons; the job is to learn the noise, trap it, and shade it away. Cross the streams only if you know what you're doing with your QPU budget. Let's bust some noise.

<div class="alert alert-block alert-warning">

__How you win: read the grader score__

Your objective is simple — get the grader score as high as you can, which means driving your mitigated value as close to the hidden reference $\langle Z_6 Z_7 \rangle$ as possible.

Submit your final number with `check_result(value)`. It returns a **0–100 score**:

- $|\text{error}| \geq 0.1$ &rarr; **0** — no better than the unmitigated (Resilience-Level-0) baseline.
- $|\text{error}| \leq 0.001$ &rarr; **100** — at the shot-noise floor.
- In between, higher is better, and a **smaller $|\text{error}|$ wins the leaderboard tiebreak**.

__Work the loop:__ apply a strategy &rarr; `check_result(value)` &rarr; read your score &rarr; adjust &rarr; resubmit. Calling `check_result` is free and you can run it as many times as you like — only *new QPU jobs* spend your budget, so reuse job IDs and iterate on the number freely.

</div>

## Background

Getting a good result out of today's QPUs is mostly about managing noise well, the errors that slip in through gate operations, state preparation, and measurement.

Three complementary strategies address it at different stages of the workflow: at compilation, during execution, and in post-processing. Circuit optimization reduces noise at the source by keeping circuits shallow — above all by minimizing two-qubit and SWAP gates, the dominant error contributors. Error suppression acts during execution, inserting control sequences that counteract coherent errors and idle-time dephasing before they accumulate. Error mitigation acts after execution, using techniques such as zero-noise extrapolation (ZNE) or probabilistic error cancellation (PEC) to post-process noisy measurements into an estimate of the noise-free result.

These strategies are complementary rather than exclusive, and a typical workflow combines all three. Note, too, that estimating an observable's expectation value requires additional measurement circuits, which introduce their own noise on top of the main circuit.


### IBM Tools

IBM provides tools covering all three strategies.

Qiskit handles circuit optimization automatically at the [transpilation](https://quantum.cloud.ibm.com/docs/en/guides/transpile) stage. For [error suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques), the [DD (Dynamical Decoupling) option](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-dynamical-decoupling-options) in the Runtime Estimator inserts protective pulse sequences during long idle periods. For [error mitigation](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-overview), lightweight methods like [ZNE](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) are cheap compared to PEC but not guaranteed to be unbiased, whereas [PEC (Probabilistic Error Cancellation)](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) represents the ideal circuit as a quasi-probability combination of noisy circuits to return an unbiased estimate — though its sampling overhead grows exponentially with circuit depth.

To push error mitigation to utility scale (40+ qubits), where plain PEC's sampling overhead becomes a huge challenge, IBM released two [add-ons](https://quantum.cloud.ibm.com/docs/en/guides/addons) that build on the per-layer noise model and cut plain PEC's sampling overhead in different ways. [SLC (Shaded Lightcones)](https://github.com/qiskit/qiskit-addon-slc) is a refinement of PEC: it keeps the quasi-probability sampling — so it stays unbiased up to a bias tolerance you set — but shades the lightcone, mitigating only the noise terms that actually influence the observable and thereby cutting the sampling overhead that limits plain PEC. [PNA (Propagated Noise Absorption)](https://github.com/qiskit/qiskit-addon-pna) takes a different route — instead of sampling, it folds the learned noise into a modified observable so that a single evaluation already returns a debiased estimate; because that noise propagation has to be truncated, the result is approximate rather than strictly unbiased (and the enlarged observable costs extra shots). Both build on the [Noise Learner](https://quantum.cloud.ibm.com/docs/en/guides/noise-learning#noiselearnerv3), IBM's tool for characterizing the noise profile of each layer and region, your key companion for this challenge.


## Your Challenge

__Before you start:__

No prior experience with these tools is assumed, but the workflow is new. SLC and PNA are recent techniques, so there is not much step-by-step material yet. The best walkthroughs today are the SLC partner lab, the [PEC with shaded lightcones](https://quantum.cloud.ibm.com/docs/en/tutorials/pec-with-shaded-lightcones) tutorial, and QGSS 2026 Lab 3, which runs Samplomatic, `NoiseLearnerV3`, the `Executor`, and both add-ons on this same kind of 1D Ising circuit. The noise model underneath all of it is the sparse Pauli–Lindblad model of [arXiv:2201.09866](https://arxiv.org/abs/2201.09866). Links are in *Useful resources* at the end.

__How the pieces fit:__

1. `NoiseLearnerV3` learns a sparse Pauli–Lindblad noise model for each unique 2-qubit layer of your circuit.
2. Samplomatic is the sampling layer. It turns the boxed circuit into a template circuit plus a *samplex*, and the samplex draws a fresh randomization for every execution: the Pauli twirls and, for PEC/SLC, the anti-noise Paulis sampled from the learned model.
3. The `Executor` runs those randomized instances on the QPU.
4. The add-ons decide what to cancel. SLC picks which noise terms are worth sampling away; PNA instead folds the noise into a modified observable.
5. Post-processing turns the bitstrings into $\langle Z_6Z_7 \rangle$ (readout correction, post-selection, and the $\gamma$ rescaling for PEC/SLC).

You are given a 15-qubit 1D transverse-field Ising model (TFIM): a Trotterized time-evolution circuit on a line of 15 qubits. Your goal is to estimate the expectation value of the central two-body observable $\langle Z_6Z_7 \rangle$ on this circuit as accurately as you can on a real QPU, then submit it to the grader.

You actually get two circuits (defined for you in *Start from here* below). `circuit` is the real, forward-only evolution: the $\langle Z_6Z_7 \rangle$ you mitigate on it is what gets scored. `circuit_mirror` is the same evolution followed by its inverse, so its ideal answer is exactly 1.0, and use it to check that your pipeline is wired up correctly before you trust the number from the real circuit. 


__Your toolbox:__

| Tool | Role |
|---|---|
| [`Samplomatic`](https://github.com/Qiskit/samplomatic/) | The sampling layer of the pipeline: boxes the circuit, then builds a (template circuit, samplex) pair; the samplex draws the twirls and anti-noise Paulis for every execution |
| [`Executor`](https://quantum.cloud.ibm.com/docs/api/qiskit-ibm-runtime/executor) | The box-aware runtime primitive (the counterpart to `Sampler`/`Estimator`) that runs the template-and-samplex program on the backend and returns the results you post-process into $\langle Z_6Z_7 \rangle$ |
| [`NoiseLearnerV3`](https://quantum.cloud.ibm.com/docs/guides/noise-learning) | Learns the sparse Pauli–Lindblad noise model of each unique 2-qubit layer |
| [`qiskit-addon-pna`](https://github.com/qiskit/qiskit-addon-pna) | Absorbs the learned noise into a modified observable (Propagated Noise Absorption) |
| [`qiskit-addon-slc`](https://qiskit.github.io/qiskit-addon-slc/) | Shades the PEC lightcone, mitigating only the noise terms that actually affect the observable and cutting sampling overhead |


<div class="alert alert-block alert-success">

__The task__

1. Learn the noise of the 15-qubit TFIM circuit with `NoiseLearnerV3`.
2. Using the learned noise model, build a mitigation strategy of your choice: PNA on its own, SLC on its own, the two together, or your own scheme (hybrids, ZNE on top, parameter tuning, layer-selective mitigation, and so on). You can also add error suppression such as dynamical decoupling (DD); suppression and mitigation are complementary. The Runtime `Estimator` has a built-in DD option. The `Executor` does not, but you can pad DD sequences into your `QuantumProgram` before running it, for example with `apply_dynamical_decoupling` from `qiskit_ibm_runtime.executor.dynamical_decoupling`. You are also free to use mitigation tools outside the IBM packages; the grader only needs your final number.
3. Validate the whole flow on `circuit_mirror` — you should recover $\langle Z_6Z_7 \rangle$ close to 1.0 — then apply the same pipeline to `circuit` and submit that mitigated estimate to the grader to see how close you land to the reference value.

</div>


There are no wrong strategies here, only results. Compare the fidelity-versus-cost trade-off of the paths you try, and be ready to say why you settled on the balance you did. Keep the notebook you actually ran and your QPU job IDs: teams presenting a final result will be asked to show that it reproduces. Time to become a Noise Buster.

## Manage resources

<div class="alert alert-block alert-info">

__Watch your QPU time__

If you are on the Open Plan, your monthly QPU allowance is small. Check your remaining time on the IBM Quantum Platform before you submit, and use `job.usage()` to see what each job actually cost. Reuse finished job IDs instead of resubmitting.

Your parameter choices are what keep the budget under your control. Noise learning is by far the biggest consumer of that budget. Its cost scales with the number of unique 2-qubit layers, times the length of `layer_pair_depths`, times `num_randomizations`. 

A reasonable starting point, tuned for under 4 minutes of learning on hardware:
- `num_randomizations = 32`, `shots_per_randomization = 128`
- `layer_pair_depths = [1, 4, 16, 32]`, with the `"edge"` post-selection strategy
- Fold repeated layers with `find_unique_box_instructions` and wrap the circuit with `twirling_strategy="active"`. This is the single biggest saving you can make.

One more option worth knowing about: [`qiskit-noise-learning`](https://github.com/Qiskit/qiskit-noise-learning). It is still beta and its API may change without notice, but its low-level interface lets you trim the learning protocol down to exactly the layers and depths your circuit needs, which can buy you the same noise information for far less learning time.

</div>


__Develop offline first — save your QPU budget__

You don't need a real backend to build and debug your pipeline. `qiskit-mitigation` is backend-agnostic (prepare circuits &rarr; run via the `Executor` &rarr; post-process classically), so the whole ZNE / PEC / PEA / TREX flow runs on a **local noisy simulator** with no QPU time and no cloud account. Wire up and debug your learn&ndash;mitigate&ndash;post-process flow there, then run only the finished pipeline on hardware for the scored submission.

Two caveats: mitigation only has something to work on with a *noisy* simulator (a noiseless one just hands back the ideal value), and simulator noise is not the real device's noise — the offline number won't match your graded QPU result.


```python
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Executor
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
import qiskit_mitigation as qm

fake = FakeSherbrooke()                          # 127-qubit fake backend, noise model built in
sim = AerSimulator.from_backend(fake)            # local, noisy, no account
isa = transpile(circuit, fake, optimization_level=1)
obs_isa = observable.apply_layout(isa.layout)

zne = qm.GateFolding()                            # ZNE via gate folding (PEC / PEA / TREX work the same way)
program = zne.prepare(circuit=isa, observables=[obs_isa], parameters=None,
                      noise_factors=(1, 3, 5), extrapolator=["linear"])

# Workaround for this version combo (qiskit-mitigation 0.1.1 + qiskit-ibm-runtime 0.50.0):
# the local Executor path raises AttributeError on `_semantic_role` without this line.
program._semantic_role = None

mitigated = zne.postprocess(Executor(mode=sim).run(program).result())
```

## Start from here

Run the cell below to get your starting point. It gives you two circuits and one observable, all fixed for everyone:

- `circuit` — the real problem. A 15-qubit 1D transverse-field Ising chain, Trotterized for `NUM_TROTTER_STEPS` steps with a transverse-field angle of `RX_ANGLE`. This is the circuit you apply your strategy to, and the mitigated $\langle Z_6 Z_7 \rangle$ you get from it is what you submit to the grader. Its exact value is not something you can read off by eye; the grader holds the reference and scores how close you land.
- `circuit_mirror` — a validation aid, not the scored circuit. It is `circuit` followed by its own inverse, so on an ideal device the state returns to the start and $\langle Z_6 Z_7 \rangle$ is exactly `1.0`. Use it to check that your pipeline is wired up correctly: run your whole learn-and-mitigate flow on `circuit_mirror` and confirm you recover something close to 1.0 before you trust the number you get from `circuit`.
- `observable` — a single `ZZ` on the two middle qubits (qubits 6 and 7 of the 15-qubit line), i.e. $\langle Z_6 Z_7 \rangle$. The same observable is used for both circuits.

> **New to the Ising model?** You don't need physics background for this. The Ising model is the generic model for binary (0/1) variables with pairwise interactions, and it shows up well beyond physics. The transverse-field version used here just adds a single-qubit $R_x$ rotation layer between the $ZZ$ interaction layers. Treat it as the benchmark circuit: what this challenge is about is the noise mitigation, not the model.

Do not change these definitions; they are what the grader expects. Everything after this cell is yours: box the circuit, learn its noise, and build your mitigation pipeline on top. Validate on `circuit_mirror`, then submit your best estimate for `circuit`.

In [ ]:
# Install dependencies. Comment these lines out after the first run.
%pip install -U "qiskit[visualization]" qiskit-ibm-runtime qiskit-aer numpy pylatexenc
%pip install -U samplomatic qiskit-addon-utils qiskit-addon-pna qiskit-addon-slc
%pip install -U qc-grader


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Connect to IBM Quantum and pick a least-busy Heron backend to run on.
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    filters=lambda b: b.processor_type.get("family") == "Heron",
)
# Reopen the backend with fractional gates enabled. The rx(π/4) kicks transpile fine
# without them, but post-selection with x_pulse_type="rx" needs rx as a native gate.
backend = service.backend(backend.name, use_fractional_gates=True)
print(f"Backend  : {backend.name}")
print(f"# qubits : {backend.num_qubits}")

from qc_grader.challenges.fallfest_2026.noise_buster import check_result

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# ---- Fixed problem definition (do not change) ----
NUM_QUBITS = 15
NUM_TROTTER_STEPS = 8
RX_ANGLE = np.pi / 4


def construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=False):
    """Trotterized 1D transverse-field Ising evolution on a line of `num_qubits`.

    Each Trotter step applies a transverse-field rotation (rx on every qubit)
    followed by the ZZ interaction on the even and then the odd bonds, realized
    as sdg-sdg + cz on each neighbouring pair.
    """
    qc = QuantumCircuit(num_qubits)
    for _ in range(num_trotter_steps):
        qc.rx(rx_angle, range(num_qubits))
        if barrier:
            qc.barrier()
        for first_qubit in (1, 2):
            for idx in range(first_qubit, num_qubits, 2):
                qc.sdg([idx - 1, idx])
                qc.cz(idx - 1, idx)
        if barrier:
            qc.barrier()
    return qc


# The real problem: forward Trotter evolution only. Apply your strategy here and
# submit the mitigated ⟨Z6Z7⟩ from THIS circuit to the grader.
circuit = construct_ising_circuit(NUM_QUBITS, NUM_TROTTER_STEPS, RX_ANGLE)

# Validation aid: the same evolution followed by its inverse (mirror)
circuit_mirror = QuantumCircuit(NUM_QUBITS)
circuit_mirror.compose(circuit, inplace=True)
circuit_mirror.barrier()
circuit_mirror.compose(circuit.inverse(), inplace=True)

# Observable: ZZ on the two middle qubits (6, 7). Same observable for both circuits.
MID = NUM_QUBITS // 2  # = 7
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", [MID - 1, MID], 1.0)], num_qubits=NUM_QUBITS
)

print(f"circuit         : {NUM_QUBITS} qubits, depth {circuit.depth()}, "
      f"{circuit.count_ops().get('cz', 0)} CZ gates  (scored)")
print(f"circuit_mirror  : depth {circuit_mirror.depth()}, "
      f"{circuit_mirror.count_ops().get('cz', 0)} CZ gates  (validation, ideal ⟨Z6Z7⟩ = 1.0)")
print(f"observable      : {observable.paulis[0]}  (Z on qubits {MID - 1} and {MID})")
circuit.draw("mpl", fold=-1, idle_wires=False, scale=0.4)

### A baseline to start from

Before any noise busting, here is the plain, unmitigated measurement of $\langle Z_6Z_7 \rangle$ on `circuit` — just transpile and run it through the `Estimator`. This is your floor: it should land well below the ideal value because of device noise, and every mitigation strategy you build should do better than this. You can submit this result to the grader right away to confirm the whole submission path works end to end, then start improving on it.

In [ ]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Transpile the scored circuit and observable onto the backend.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
circuit_isa = pm.run(circuit)
observable_isa = observable.apply_layout(circuit_isa.layout)

# Plain, unmitigated expectation value: no PNA, no SLC, resilience level 0.
estimator = Estimator(mode=backend)
estimator.options.resilience_level = 0
estimator.options.default_shots = 10_000

baseline_job = estimator.run([(circuit_isa, observable_isa)])


In [ ]:
job_id = baseline_job.job_id()
print(job_id)
baseline_job.status()

In [ ]:
baseline_result = service.job(job_id).result()
baseline_value = float(baseline_result[0].data.evs)  # evs is a 0-d array for a single pub
print(f"Unmitigated ⟨Z6Z7⟩ = {baseline_value:.4f}")

In [ ]:
# Sanity-check the submission path with this baseline, then beat it.
# check_result takes a plain Python float on purpose: your pipeline can end in any
# tool (IBM add-ons, other mitigation libraries, your own code), so the grader only
# asks for the final mitigated <Z6 Z7> value.
check_result(baseline_value)

### You're off — now build your strategy

That baseline is your starting line, not your finish. Everything below is yours to write.

Explore the mitigation techniques on the table — TREX, ZNE, PEC, and the SLC / PNA add-ons, plus error suppression such as dynamical decoupling — and get a feel for what each one buys you and what it costs: extra measurement circuits, sampling overhead ($\gamma^2$), bias versus variance, and QPU time. Then combine them into the strategy that drives this circuit's $\langle Z_6 Z_7 \rangle$ closest to the reference.

Apply your strategy to the target `circuit` and push for the highest score you can. Pass the value you obtain to `check_result(value)` to get your score.

### Useful resources

**Start here for this challenge**
- SLC partner lab (`SLC_magnetization_partner_lab.ipynb`): the guided, graded version of this pipeline.
- [Probabilistic error cancellation with shaded lightcones](https://quantum.cloud.ibm.com/docs/en/tutorials/pec-with-shaded-lightcones) (tutorial)
- [Improve expectation values with propagated noise absorption (PNA)](https://quantum.cloud.ibm.com/docs/en/tutorials/propagated-noise-absorption) (tutorial)
- [QGSS 2026 Lab 3](https://github.com/qiskit-community/qgss-2026/blob/main/lab-3/QGSS2026_Lab3.ipynb): Samplomatic, `NoiseLearnerV3`, the `Executor`, and both add-ons on the same kind of 1D Ising circuit.
<!-- TODO: add the new IBM Quantum Learning module here once it is released -->
- QDC 2025 SLC & PNA challenge ([repo](https://github.com/qiskit-community/qdc-challenges-2025) · [blog](https://www.ibm.com/quantum/blog/qdc-2025))
- Add-on docs: [SLC](https://qiskit.github.io/qiskit-addon-slc/) · [PNA](https://qiskit.github.io/qiskit-addon-pna/)

**New to quantum computing**
- [Hello world](https://quantum.cloud.ibm.com/docs/en/guides/hello-world)
- [Quickstart](https://quantum.cloud.ibm.com/docs/en/guides/quick-start)
- [Quantum Computing in Practice](https://quantum.cloud.ibm.com/learning/en/courses/quantum-computing-in-practice) (course)
- [Use a Quantum Computer Today](https://quantum.cloud.ibm.com/learning/en/courses/use-a-qc-today) (course)

**Videos**
- [Why Quantum? What Problem Are We Solving?](https://www.youtube.com/watch?v=yqDYNjwTUSI)
- [Noise, Hardware, and Reality](https://www.youtube.com/watch?v=0z0kNUkxXNU)
- [Error Mitigation Landscape](https://www.youtube.com/watch?v=ix52wx4_zek)

**More error mitigation**
- [Tools for Quantum Advantage](https://quantum.cloud.ibm.com/learning/en/courses/tools-for-quantum-advantage) (course)
- [Probabilistic error cancellation with logical noise models](https://quantum.cloud.ibm.com/docs/en/tutorials/probabilistic-error-cancellation-with-logical-noise-models)
- [Utility-scale error mitigation with probabilistic error amplification](https://quantum.cloud.ibm.com/docs/en/tutorials/probabilistic-error-amplification)
- [Combine error mitigation options with the Estimator primitive](https://quantum.cloud.ibm.com/docs/en/tutorials/combine-error-mitigation-techniques)
- [Real-time benchmarking for qubit selection](https://quantum.cloud.ibm.com/docs/en/tutorials/real-time-benchmarking-for-qubit-selection) · [Refresh backend properties with real-time benchmarking](https://quantum.cloud.ibm.com/docs/en/tutorials/refresh-backend-properties-with-real-time-benchmarking): picking good qubits before you run.

**Other tools** (you are free to use mitigation tools outside the IBM packages)
- [Mitiq](https://github.com/unitaryfoundation/mitiq): open-source ZNE, PEC, CDR and more
- [M3](https://github.com/Qiskit/qiskit-addon-mthree): matrix-free readout error mitigation
- [Qiskit ecosystem: error mitigation projects](https://www.ibm.com/quantum/ecosystem?tag=error+mitigation)
- [Awesome quantum software](https://github.com/qosf/awesome-quantum-software)

**Papers**
- van den Berg et al., *Probabilistic error cancellation with sparse Pauli–Lindblad models on noisy quantum processors*, [arXiv:2201.09866](https://arxiv.org/abs/2201.09866). The noise model that `NoiseLearnerV3`, PEC, SLC and PNA all build on.
- Eddins et al., *Lightcone shading for classically accelerated quantum error mitigation*, [arXiv:2409.04401](https://arxiv.org/abs/2409.04401). The theory behind SLC.
- Kim et al., *Evidence for the utility of quantum computing before fault tolerance*, [Nature 618, 500 (2023)](https://www.nature.com/articles/s41586-023-06096-3). Noise learning and mitigation on the same kicked-Ising circuit family at 127 qubits.